In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from pathlib import Path
from google.colab import userdata

# Download Hadoop directly from GitHub
!git clone https://github.com/apache/hadoop.git

In [ ]:
SOURCE_CODE_DIR = Path("/content/hadoop/hadoop-mapreduce-project/hadoop-mapreduce-client/hadoop-mapreduce-client-core/src/main/java/org/apache/hadoop/mapreduce")

files_list = []      # Stores the actual Java code
file_names = []      # Stores the name of the file (important for the RSF match later)

# Search for all Java files in the core folder
for file_path in SOURCE_CODE_DIR.rglob('*.java'):
    if file_path.is_file():
        with open(file_path, 'r', encoding='utf-8') as f:
            files_list.append(f.read())
            file_names.append(file_path.stem)

print(f"Successfully loaded {len(files_list)} Java files.")

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

In [ ]:
# Change this to "jinaai/jina-code-embeddings-0.5b" when you do your second run!
embedding_model_name = "ibm-granite/granite-embedding-english-r2"
# embedding_model_name = "jinaai/jina-code-embeddings-0.5b"

tokenizer = AutoTokenizer.from_pretrained(embedding_model_name, token=hf_token, trust_remote_code=True)
model = AutoModel.from_pretrained(embedding_model_name, token=hf_token, trust_remote_code=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Loaded {embedding_model_name} onto {device}")

In [ ]:
def embed_source_code(code_files):
    embeddings = []
    for code in code_files:
        # Cap the length so it doesn't crash Colab memory
        inputs = tokenizer(code, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # Extract the pure math vector (Mean Pooling)
            token_embeddings = outputs.last_hidden_state
            attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * attention_mask, 1)
            sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
            vector = (sum_embeddings / sum_mask).cpu().numpy()

        embeddings.append(vector[0])
    return np.array(embeddings)

print("Reading code and generating vectors... (This may take a few minutes)")
embeddings = embed_source_code(files_list)
semantic_matrix = cosine_similarity(embeddings)
print("Semantic Matrix Complete!")

**3. Hardware optimization (Quantization)**

In [ ]:
num_files = len(file_names)
struct_matrix_raw = np.zeros((num_files, num_files))

# Create a dictionary to map a class name to its row in the matrix
name_to_index = {name: idx for idx, name in enumerate(file_names)}

# Point this to the file you uploaded!
RSF_FILE_PATH = "/content/mapreduce_filtered.rsf"

try:
    with open(RSF_FILE_PATH, 'r') as file:
        for line in file:
            parts = line.strip().split()
            # If the RSF says File A "depends" on File B
            if len(parts) >= 3 and parts[0] == "depends":
                class_a = parts[1].split('.')[-1]
                class_b = parts[2].split('.')[-1]

                # If we have both files, log the connection
                if class_a in name_to_index and class_b in name_to_index:
                    idx_a = name_to_index[class_a]
                    idx_b = name_to_index[class_b]
                    struct_matrix_raw[idx_a][idx_b] += 1
                    struct_matrix_raw[idx_b][idx_a] += 1

    print("Structural Matrix successfully built from RSF!")
except FileNotFoundError:
    print(f"ERROR: Could not find {RSF_FILE_PATH}. Make sure you uploaded it to the left sidebar.")

In [ ]:
max_overlap = struct_matrix_raw.max()
struct_matrix = struct_matrix_raw / max_overlap if max_overlap > 0 else struct_matrix_raw
np.fill_diagonal(struct_matrix, 1.0)
print("Structural Normalization Complete!")

In [ ]:
# Combine 50% Structure with 50% Semantic Meaning
ALPHA = 0.5
combined_similarity = (ALPHA * struct_matrix) + ((1 - ALPHA) * semantic_matrix)

# Invert to Distance (Clustering needs distance, not similarity)
distance_matrix = 1.0 - combined_similarity
np.fill_diagonal(distance_matrix, 0)


# Run the final agglomerative clustering
TARGET_NUM_CLUSTERS = 10
clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, metric='precomputed', linkage='complete')
clusters = clusterer.fit_predict(distance_matrix)

print(f"SUCCESS! Clustered {len(files_list)} Java files into {TARGET_NUM_CLUSTERS} groups.")

In [ ]:
print(distance_matrix)

In [ ]:
# Create a new output file
OUTPUT_FILE = "/content/week3_arc_jinaaiclusters.rsf"

with open(OUTPUT_FILE, "w") as f:
    # Loop through the files and the cluster numbers assigned to them
    for i in range(len(file_names)):
        cluster_id = clusters[i]
        file_name = file_names[i]

        # Write it in the standard RSF format: "contain cluster_ID File_Name"
        f.write(f"contain cluster_{cluster_id} {file_name}\n")

print(f"SUCCESS! Your output has been saved to: {OUTPUT_FILE}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.manifold import TSNE

print("Generating the improved 2D Architecture Map...")

# 1. Squash the complex math into a flat 2D layout
tsne = TSNE(n_components=2, metric='precomputed', random_state=42, init='random', perplexity=5)
coords_2d = tsne.fit_transform(distance_matrix)

# 2. Package the data cleanly
df = pd.DataFrame({
    'Distance X': coords_2d[:, 0],
    'Distance Y': coords_2d[:, 1],
    'Cluster': clusters,
    'File': file_names
})

# 3. Create a clean legend label.
# We only show the top 15 biggest clusters in the legend, otherwise it ruins the image!
top_clusters = df['Cluster'].value_counts().nlargest(15).index
df['Legend_Label'] = df['Cluster'].apply(lambda c: f"Cluster {c}" if c in top_clusters else "Other Small Clusters")

# 4. Create the final, clean plot
plt.figure(figsize=(14, 9))
sns.scatterplot(
    data=df,
    x='Distance X',
    y='Distance Y',
    hue='Legend_Label',
    palette='tab20',
    s=120,           # Slightly larger dots
    alpha=0.9,
    edgecolor='black'
)

# 5. Move the legend OUTSIDE the graph box
plt.legend(title="Largest Architectural Subsystems", bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0, fontsize=11, title_fontsize=13)

# 6. Proper, understandable titles
plt.title("Hadoop MapReduce: Codebase Architecture Map", fontsize=18, pad=10)
plt.suptitle("How to read this: Dots closer together = Highly connected code. Dots far apart = Unrelated code.", fontsize=12, color='gray')

# 7. Remove the actual axis numbers and lines because the literal X/Y coordinates don't mean anything
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.xlabel("")
plt.ylabel("")

plt.tight_layout()
plt.show()
print("Visualization Complete!")